# Continuous Audit — Planner Notifier
Cria um card no bucket STAND-BY para cada novo achado/reincidente. Dedup pelo
próprio Planner: card ABERTO com o mesmo título é atualizado, nunca duplicado
(sem estado local; renomear o título quebra o vínculo — manter o prefixo).
Credenciais Graph nos secrets `planner-*` do scope `compliance-grc`.
Consumido pelo orquestrador via `%run` após o `utils`.


In [ ]:
import requests
from datetime import datetime
from zoneinfo import ZoneInfo

_BRT   = ZoneInfo("America/Sao_Paulo")
_GRAPH = "https://graph.microsoft.com/v1.0"

# ── Config do Planner (resolvida por NOME a cada rodada — sobrevive a mudanças de ID)
_PLANNER_GROUP_ID  = "b5d2a8c9-e93c-4f88-b09b-0aca8f9c7147"   # grupo GRC
_PLANNER_PLAN_NAME = "Planner_GRC"
_PLANNER_BUCKET    = "STAND-BY"                                # cards novos SEMPRE nascem aqui
_LABEL_HINTS       = ("gestão de riscos", "dedo no pulso")     # labels rosa + roxa
_TITLE_PREFIX      = "[Continuous Audit] "

_CHECKLIST = [
    "Analisar os achados no painel e marcar falsos positivos, se houver",
    "Acionar a área responsável e definir o plano de ação",
    "Se for a tratamento: emitir e vincular o apontamento no painel — o alerta fica suprimido até o apontamento fechar",
    "Reavaliar após as ações e confirmar rodada sem achados",
]

_ALERT_LABEL = {"novo_achado": "Novo achado", "reincidente": "Reincidente"}


def _graph_headers():
    tenant = dbutils.secrets.get("compliance-grc", "planner-tenant-id")
    cid    = dbutils.secrets.get("compliance-grc", "planner-client-id")
    csec   = dbutils.secrets.get("compliance-grc", "planner-client-secret")
    r = requests.post(
        f"https://login.microsoftonline.com/{tenant}/oauth2/v2.0/token",
        data={"client_id": cid, "client_secret": csec,
              "grant_type": "client_credentials",
              "scope": "https://graph.microsoft.com/.default"},
        timeout=20,
    ).json()
    if "access_token" not in r:
        raise RuntimeError(f"Token Graph falhou: {str(r)[:200]}")
    return {"Authorization": f"Bearer {r['access_token']}",
            "Content-Type": "application/json"}


def _resolve_plan(H):
    plans   = requests.get(f"{_GRAPH}/groups/{_PLANNER_GROUP_ID}/planner/plans",
                           headers=H, timeout=20).json()["value"]
    plan    = next(p for p in plans if p["title"] == _PLANNER_PLAN_NAME)
    buckets = requests.get(f"{_GRAPH}/planner/plans/{plan['id']}/buckets",
                           headers=H, timeout=20).json()["value"]
    bucket  = next(b for b in buckets
                   if b["name"].strip().upper() == _PLANNER_BUCKET.upper())
    cats    = (requests.get(f"{_GRAPH}/planner/plans/{plan['id']}/details",
                            headers=H, timeout=20).json()
               .get("categoryDescriptions") or {})
    labels  = {k: True for k, v in cats.items()
               if v and any(h in v.lower() for h in _LABEL_HINTS)}
    return plan["id"], bucket["id"], (labels or {"category1": True, "category6": True})


def _open_tasks_by_title(H, plan_id):
    """Dedup direto na fonte: lista os cards ABERTOS do plano (percentComplete < 100)
    com o prefixo do sistema, indexados por título. Enxerga inclusive cards criados
    manualmente — sem estado local, sem tabela."""
    abertos, url = {}, f"{_GRAPH}/planner/plans/{plan_id}/tasks"
    while url:
        r = requests.get(url, headers=H, timeout=30).json()
        for t in r.get("value", []):
            if (t.get("percentComplete", 100) < 100
                    and (t.get("title") or "").startswith(_TITLE_PREFIX)):
                abertos.setdefault(t["title"], t)
        url = r.get("@odata.nextLink")
    return abertos


def _description(e, risk, app_url):
    linhas = ["Trigger da Auditoria Contínua — achados que exigem análise.", "",
              f"Teste: {e['test_name']}"]
    if e.get("description"):
        linhas.append(f"O que o teste verifica: {e['description']}")
    if e.get("risco_id") and e["risco_id"] != "N/A":
        extra = ""
        if risk.get("title"):
            extra += f" — {risk['title']}"
        if risk.get("level"):
            extra += f" (Inerente: {risk['level']})"
        linhas.append(f"Risco: {e['risco_id']}{extra}")
    if e.get("area"):
        linhas.append(f"Área responsável: {e['area']}")
    linhas.append(f"Tipo de alerta: {_ALERT_LABEL.get(e['alert'], e['alert'])}")
    linhas.append(f"Achados na rodada: {e['count']}")
    linhas.append(f"Rodada: {datetime.now(_BRT).strftime('%d/%m/%Y %H:%M')} (BRT)")
    if app_url:
        linhas += ["", f"Painel: {app_url}"]
    return "\n".join(linhas)


def notify_planner_cards(events, risk_info=None, app_url=None) -> int:
    """Cria/atualiza cards no Planner para novos achados e reincidentes.

    Dedup pelo próprio Planner (sem estado local): se já existe um card ABERTO
    com o título "[Continuous Audit] {teste}", ele é atualizado — nunca
    duplicado (mesmo que o time o tenha movido de bucket). Card concluído ou
    excluído → novo trigger abre card novo (episódio novo). Renomear o título
    de um card quebra o vínculo — manter o prefixo.
    Erros de execução não viram card (problema técnico, não risco).
    """
    risk_info = risk_info or {}
    dedup = {}
    for e in events:
        dedup[e["test_name"]] = e
    cards = [e for e in dedup.values()
             if e["alert"] in ("novo_achado", "reincidente") and e["notify"]]
    if not cards:
        print("Planner: nenhum trigger para card.")
        return 0

    H = _graph_headers()
    plan_id, bucket_id, labels = _resolve_plan(H)
    abertos = _open_tasks_by_title(H, plan_id)

    criados = atualizados = 0
    for e in cards:
        titulo = f"{_TITLE_PREFIX}{e['test_name']}"
        risk   = risk_info.get(e.get("risco_id"), {})
        desc   = _description(e, risk, app_url)

        existente = abertos.get(titulo)
        if existente:
            det = requests.get(f"{_GRAPH}/planner/tasks/{existente['id']}/details",
                               headers=H, timeout=20)
            requests.patch(f"{_GRAPH}/planner/tasks/{existente['id']}/details",
                           headers={**H, "If-Match": det.json()["@odata.etag"]},
                           json={"description": desc}, timeout=20)
            atualizados += 1
            print(f"Planner ↻ card aberto atualizado: {e['test_name']}")
            continue

        t = requests.post(f"{_GRAPH}/planner/tasks", headers=H, json={
            "planId": plan_id, "bucketId": bucket_id,
            "title": titulo,
            "priority": 3,
            "appliedCategories": labels,
        }, timeout=20).json()
        if "id" not in t:
            print(f"Planner: falha ao criar card de {e['test_name']}: {str(t)[:150]}")
            continue
        det = requests.get(f"{_GRAPH}/planner/tasks/{t['id']}/details",
                           headers=H, timeout=20)
        checklist = {str(i): {"@odata.type": "microsoft.graph.plannerChecklistItem",
                              "title": s, "isChecked": False}
                     for i, s in enumerate(_CHECKLIST, 1)}
        requests.patch(f"{_GRAPH}/planner/tasks/{t['id']}/details",
                       headers={**H, "If-Match": det.json()["@odata.etag"]},
                       json={"description": desc, "checklist": checklist,
                             "previewType": "checklist"}, timeout=20)
        criados += 1
        print(f"Planner + card criado: {e['test_name']}")

    print(f"Planner: {criados} criado(s) · {atualizados} atualizado(s).")
    return criados
